# **1. Installs/Imports**


### huggingface.co Read Token Required

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from google.colab import drive

### Optional: Mount Google Drive

In [ ]:
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/commonsenseqa_paraphrase_evaluation")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Google Drive save folder: {PROJECT_DIR}")

def save_df_to_drive(df, filename, index=False):
    path = PROJECT_DIR / filename
    df.to_csv(path, index=index)
    print(f"Saved -> {path}")
    return path

def save_plot_to_drive(filename, dpi=300, bbox_inches="tight"):
    path = PROJECT_DIR / filename
    plt.savefig(path, dpi=dpi, bbox_inches=bbox_inches)
    print(f"Saved plot -> {path}")
    return path

### Loading CommonsenseQA

In [ ]:
dataset = load_dataset("tau/commonsense_qa")

SPLIT_NAME = "train"
ds = dataset[SPLIT_NAME]

print(ds)
print("Number of rows:", len(ds))

### Base Dataframe - For Analysis

In [ ]:
# Format Dataset for experiments
def extract_choices(choice_dict):
    labels = choice_dict["label"]
    texts = choice_dict["text"]
    mapping = {label: text for label, text in zip(labels, texts)}
    return mapping

records = []
for row in ds:
    choices = extract_choices(row["choices"])
    records.append({
        "id": row.get("id", ""),
        "question": row["question"],
        "choice_A": choices.get("A", ""),
        "choice_B": choices.get("B", ""),
        "choice_C": choices.get("C", ""),
        "choice_D": choices.get("D", ""),
        "choice_E": choices.get("E", ""),
        "answerKey": row.get("answerKey", "")
    })

base_df = pd.DataFrame(records)
base_df.head()

save_df_to_drive(base_df, "base_df.csv") --------- OPTIONAL SAVE TO GOOGLE DRIVE

### Helper Function

In [ ]:
# Minimum word count for each question
def keep_question(question: str, min_words: int = 8) -> bool:
    return len(question.split()) >= min_words

### Filtered Dataframe

In [ ]:
MIN_WORDS = 8 # Minimum words per paraphrase

filtered_df = base_df[base_df["question"].apply(lambda q: keep_question(q, MIN_WORDS))].copy() # Minimum word per paraphrase filter
filtered_df = filtered_df.sample(n=2000, random_state=42).reset_index(drop=True) # Sample of 2000 questions

print("Original Rows:", len(base_df))
print("Filtered Rows:", len(filtered_df))

save_df_to_drive(filtered_df, "filtered_df.csv") --------- OPTIONAL SAVE TO GOOGLE DRIVE